## Pipeline — Hierarchický ML pipeline

Pipeline načítava natrénované modely zo súborov `models/` a spúšťa ich sekvenčne na testovacej množine:

1. **Model 1** — klasifikuje všetky záznamy na normálnu prevádzku a poruchu
2. **Model 2** — z záznamov klasifikovaných ako porucha identifikuje minoritné typy
3. **Model 3** — z minoritných záznamov klasifikuje konkrétny typ poruchy

Výsledky pipeline sa ukladajú do súboru `results/pipeline_3models_predictions.csv`.

> **Pred spustením** je potrebné mať natrénované všetky tri modely a uložené súbory `models/model1.pkl`, `models/model2.pkl` a `models/model3_models.pkl`

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from collections import defaultdict
from sklearn.metrics import (
    classification_report, confusion_matrix,
    recall_score, precision_score, f1_score
)


DATA_PATH   = 'data/final_typy.csv'
MODELS_DIR  = 'models'


# Načítanie a zoradenie dát
df = pd.read_csv(DATA_PATH)
df['t_utc'] = pd.to_datetime(df['t_utc'])
df_sorted = df.sort_values(['eic', 't_utc']).reset_index(drop=True)

fault_cols = [c for c in df_sorted.columns if c.startswith('fault_type')]

# Rozdelenie podľa EIC — rovnaký kód ako v modeloch
eic_fault_profile = {}
for eic in sorted(df_sorted['eic'].unique()):
    eic_data = df_sorted[df_sorted['eic'] == eic]
    profile  = tuple(int(eic_data[col].sum() > 0) for col in sorted(fault_cols))
    eic_fault_profile[eic] = profile

profile_groups = defaultdict(list)
for eic, profile in eic_fault_profile.items():
    profile_groups[profile].append(eic)

np.random.seed(42)
train_eics, test_eics = [], []

for profile, eics in sorted(profile_groups.items()):
    eics_shuffled = eics.copy()
    np.random.shuffle(eics_shuffled)
    if len(eics_shuffled) == 1:
        train_eics.extend(eics_shuffled)
    elif len(eics_shuffled) == 2:
        train_eics.append(eics_shuffled[0])
        test_eics.append(eics_shuffled[1])
    else:
        split_n = max(1, int(len(eics_shuffled) * 0.8))
        train_eics.extend(eics_shuffled[:split_n])
        test_eics.extend(eics_shuffled[split_n:])

train_eics = set(train_eics)
test_eics  = set(test_eics)

train_df = df_sorted[df_sorted['eic'].isin(train_eics)].copy()
test_df  = df_sorted[df_sorted['eic'].isin(test_eics)].copy()

# Definícia príznakov
feature_cols = [
    'i1_norm_max', 'i1_norm_mean', 'i1_norm_min',
    'i2_norm_max', 'i2_norm_mean', 'i2_norm_min',
    'i3_norm_max', 'i3_norm_mean', 'i3_norm_min',
    'u1_norm_min', 'u1_norm_max', 'u1_norm_mean',
    'u2_norm_min', 'u2_norm_max', 'u2_norm_mean',
    'u3_norm_min', 'u3_norm_max', 'u3_norm_mean',
    'u1_norm_p2p', 'u1_norm_std', 'u1_norm_mean_abs_diff',
    'u2_norm_p2p', 'u2_norm_mean_abs_diff',
    'u3_norm_p2p', 'u3_norm_mean_abs_diff',
    'u1_norm_kurtosis',
]

MINORITY_COLS = [
    'fault_type_7',
    'fault_type_8',
    'fault_type_9',
    'fault_type_13',
]

def add_fault_profile_features(df):
    """
    Výpočet profilových príznakov zo surových signálov.
    Zachytáva medzifázové rozdiely a celkové charakteristiky signálov.
    """
    df  = df.copy()
    eps = 1e-6

    i_max_cols  = ['i1_norm_max',  'i2_norm_max',  'i3_norm_max']
    i_mean_cols = ['i1_norm_mean', 'i2_norm_mean', 'i3_norm_mean']
    i_min_cols  = ['i1_norm_min',  'i2_norm_min',  'i3_norm_min']
    u_max_cols  = ['u1_norm_max',  'u2_norm_max',  'u3_norm_max']
    u_mean_cols = ['u1_norm_mean', 'u2_norm_mean', 'u3_norm_mean']
    u_min_cols  = ['u1_norm_min',  'u2_norm_min',  'u3_norm_min']
    u_p2p_cols  = ['u1_norm_p2p',  'u2_norm_p2p',  'u3_norm_p2p']
    u_mad_cols  = ['u1_norm_mean_abs_diff', 'u2_norm_mean_abs_diff', 'u3_norm_mean_abs_diff']

    df['profile_i_max_spread']  = df[i_max_cols].max(axis=1)  - df[i_max_cols].min(axis=1)
    df['profile_i_mean_spread'] = df[i_mean_cols].max(axis=1) - df[i_mean_cols].min(axis=1)
    df['profile_i_min_spread']  = df[i_min_cols].max(axis=1)  - df[i_min_cols].min(axis=1)
    df['profile_u_max_spread']  = df[u_max_cols].max(axis=1)  - df[u_max_cols].min(axis=1)
    df['profile_u_mean_spread'] = df[u_mean_cols].max(axis=1) - df[u_mean_cols].min(axis=1)
    df['profile_u_min_spread']  = df[u_min_cols].max(axis=1)  - df[u_min_cols].min(axis=1)
    df['profile_i_mean_total']  = df[i_mean_cols].mean(axis=1)
    df['profile_u_mean_total']  = df[u_mean_cols].mean(axis=1)
    df['profile_i_max_ratio']   = df[i_max_cols].max(axis=1) / (df[i_max_cols].min(axis=1).abs() + eps)
    df['profile_u_max_ratio']   = df[u_max_cols].max(axis=1) / (df[u_max_cols].min(axis=1).abs() + eps)
    df['profile_u_p2p_mean']    = df[u_p2p_cols].mean(axis=1)
    df['profile_u_p2p_spread']  = df[u_p2p_cols].max(axis=1) - df[u_p2p_cols].min(axis=1)
    df['profile_u_mad_mean']    = df[u_mad_cols].mean(axis=1)
    df['profile_u_mad_spread']  = df[u_mad_cols].max(axis=1) - df[u_mad_cols].min(axis=1)

    return df

# Mediány z trénovacej množiny
train_medians_model1 = train_df[feature_cols].median()
train2               = train_df[train_df['porucha?'] == 1].copy()
train_medians_model2 = train2[feature_cols].median()
train3               = train_df[train_df[MINORITY_COLS].max(axis=1) == 1].copy()
train3               = add_fault_profile_features(train3)
profile_cols         = [c for c in train3.columns if c.startswith('profile_')]
feature_cols_model3  = feature_cols + profile_cols
train_medians_model3 = train3[feature_cols_model3].median()

# Načítanie natrenovaných modelov zo súborov
model1        = joblib.load(f'{MODELS_DIR}/model1.pkl')
model2        = joblib.load(f'{MODELS_DIR}/model2.pkl')
model3_models = joblib.load(f'{MODELS_DIR}/model3_models.pkl')

print(f"Načítané modely zo súborov v priečinku {MODELS_DIR}/")

# Prahové hodnoty
THRESHOLD  = 0.7   # Model 1
THRESHOLD2 = 0.25  # Model 2

# Prahové hodnoty pre Model 3 — zapísané po finálnom spustení
model3_thresholds = {
    'fault_type_7':  0.40,
    'fault_type_8':  0.85,
    'fault_type_9':  0.45,
    'fault_type_13': 0.09,
}

feature_cols_model1 = feature_cols
feature_cols_model2 = feature_cols
trained_labels      = list(model3_models.keys())

# Príprava vstupných dát z testovacej množiny
X_pipe = test_df[feature_cols_model1].fillna(train_medians_model1)

y_true_fault = test_df["porucha?"].astype(int)
y_true_minor = (test_df[MINORITY_COLS].sum(axis=1) > 0).astype(int)
Y_true_types = test_df[MINORITY_COLS].astype(int)

# Model 1 — detekcia poruchy
prob_m1 = model1.predict_proba(X_pipe)[:, 1]
pred_m1 = (prob_m1 >= THRESHOLD).astype(int)

# Model 2 — minoritná / majoritná porucha
# Spúšťa sa iba pre záznamy kde Model 1 detekoval poruchu
pred_m2 = np.zeros(len(test_df), dtype=int)
prob_m2 = np.zeros(len(test_df), dtype=float)

idx_m1_fault = np.where(pred_m1 == 1)[0]

if len(idx_m1_fault) > 0:
    X_m2     = test_df.iloc[idx_m1_fault][feature_cols_model2].fillna(train_medians_model2)
    prob_tmp = model2.predict_proba(X_m2)[:, 1]
    pred_tmp = (prob_tmp >= THRESHOLD2).astype(int)

    prob_m2[idx_m1_fault] = prob_tmp
    pred_m2[idx_m1_fault] = pred_tmp

# Model 3 — klasifikácia konkrétneho typu minoritnej poruchy
# Spúšťa sa iba pre záznamy kde Model 1 = porucha a Model 2 = minoritná
Y_pred_types = pd.DataFrame(0,   index=test_df.index, columns=trained_labels)
Y_prob_types = pd.DataFrame(0.0, index=test_df.index, columns=trained_labels)

idx_m2_minor = np.where((pred_m1 == 1) & (pred_m2 == 1))[0]

if len(idx_m2_minor) > 0:
    X_m3_raw = test_df.iloc[idx_m2_minor].copy()
    X_m3_p   = add_fault_profile_features(X_m3_raw)
    X_m3     = X_m3_p[feature_cols_model3].fillna(train_medians_model3)

    for label in trained_labels:
        prob_l = model3_models[label].predict_proba(X_m3)[:, 1]
        pred_l = (prob_l >= model3_thresholds[label]).astype(int)

        Y_prob_types.loc[test_df.index[idx_m2_minor], label] = prob_l
        Y_pred_types.loc[test_df.index[idx_m2_minor], label] = pred_l

# Zostavenie výslednej tabuľky predikcií
pipeline_result = test_df[["eic", "t_utc", "porucha?"]].copy()

pipeline_result["prob_fault_m1"] = prob_m1
pipeline_result["pred_fault_m1"] = pred_m1
pipeline_result["prob_minor_m2"] = prob_m2
pipeline_result["pred_minor_m2"] = pred_m2

for label in trained_labels:
    pipeline_result[f"pred_{label}"] = Y_pred_types[label].values
    pipeline_result[f"prob_{label}"] = Y_prob_types[label].values

# Report — Model 1
print("=" * 70)
print("Report — Model 1: normálna prevádzka / porucha")
print("=" * 70)
print(classification_report(
    y_true_fault, pred_m1,
    target_names=["Normálna prevádzka", "Porucha"],
    zero_division=0
))
cm1 = confusion_matrix(y_true_fault, pred_m1)
print("Matica zámen — Model 1:")
print(cm1)

# Report — detekcia minoritnej poruchy cez celý pipeline
pipeline_pred_minor = ((pred_m1 == 1) & (pred_m2 == 1)).astype(int)

print("\n" + "=" * 70)
print("Report — detekcia minoritnej poruchy: celý pipeline")
print("=" * 70)
print(classification_report(
    y_true_minor, pipeline_pred_minor,
    target_names=["Majoritná porucha", "Minoritná porucha"],
    zero_division=0
))
cm_minor = confusion_matrix(y_true_minor, pipeline_pred_minor)
print("Matica zámen — minoritná porucha:")
print(cm_minor)
print(f"Návratnosť (recall):  {recall_score(y_true_minor, pipeline_pred_minor, zero_division=0):.4f}")
print(f"Presnosť (precision): {precision_score(y_true_minor, pipeline_pred_minor, zero_division=0):.4f}")
print(f"F1:                   {f1_score(y_true_minor, pipeline_pred_minor, zero_division=0):.4f}")

# Report — klasifikácia konkrétnych typov minoritných porúch
Y_true_eval = Y_true_types[trained_labels].astype(int)
Y_pred_eval = Y_pred_types[trained_labels].astype(int)

print("\n" + "=" * 70)
print("Report — konkrétne typy minoritných porúch")
print("=" * 70)
print(classification_report(
    Y_true_eval, Y_pred_eval,
    target_names=trained_labels,
    zero_division=0
))
print("Micro metriky:")
print(f"  Micro návratnosť (recall):  {recall_score(Y_true_eval, Y_pred_eval, average='micro', zero_division=0):.4f}")
print(f"  Micro presnosť (precision): {precision_score(Y_true_eval, Y_pred_eval, average='micro', zero_division=0):.4f}")
print(f"  Micro F1:                   {f1_score(Y_true_eval, Y_pred_eval, average='micro', zero_division=0):.4f}")

# Matice zámen pre každý typ poruchy
print("\n" + "=" * 70)
print("Matice zámen — jednotlivé typy minoritných porúch")
print("=" * 70)

for label in trained_labels:
    cm          = confusion_matrix(Y_true_eval[label], Y_pred_eval[label], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{label}")
    print(f"  Správne negatívne (TN):  {tn}")
    print(f"  Falošne pozitívne (FP):  {fp}")
    print(f"  Falošne negatívne (FN):  {fn}")
    print(f"  Správne pozitívne (TP):  {tp}")
    print(f"  Návratnosť (recall):     {tp / max(tp + fn, 1):.4f}")
    print(f"  Presnosť (precision):    {tp / max(tp + fp, 1):.4f}")



## Výsledky — Pipeline

Načítané modely zo súborov v priečinku models/
======================================================================
Report — Model 1: normálna prevádzka / porucha
======================================================================
                    precision    recall  f1-score   support

Normálna prevádzka       0.71      0.79      0.75     18736
           Porucha       0.88      0.83      0.86     35642

          accuracy                           0.82     54378
         macro avg       0.80      0.81      0.80     54378
      weighted avg       0.83      0.82      0.82     54378

Matica zámen — Model 1:
[[14845  3891]
 [ 5942 29700]]

======================================================================
Report — detekcia minoritnej poruchy: celý pipeline
======================================================================
                   precision    recall  f1-score   support

Majoritná porucha       0.93      0.94      0.93     47905
Minoritná porucha       0.51      0.48      0.50      6473

         accuracy                           0.88     54378
        macro avg       0.72      0.71      0.72     54378
     weighted avg       0.88      0.88      0.88     54378

Matica zámen — minoritná porucha:
[[44963  2942]
 [ 3367  3106]]
Návratnosť (recall):  0.4798
Presnosť (precision): 0.5136
F1:                   0.4961

======================================================================
Report — konkrétne typy minoritných porúch
======================================================================
               precision    recall  f1-score   support

 fault_type_7       0.35      0.37      0.36      2780
 fault_type_8       0.61      0.69      0.65      2608
 fault_type_9       0.61      0.71      0.65      3068
fault_type_13       0.00      0.00      0.00       951

    micro avg       0.46      0.53      0.49      9407
    macro avg       0.39      0.44      0.42      9407
 weighted avg       0.47      0.53      0.50      9407
  samples avg       0.06      0.06      0.06      9407

Micro metriky:
  Micro návratnosť (recall):  0.5311
  Micro presnosť (precision): 0.4607
  Micro F1:                   0.4934

======================================================================
Matice zámen — jednotlivé typy minoritných porúch
======================================================================

fault_type_7
  Správne negatívne (TN):  49664
  Falošne pozitívne (FP):  1934
  Falošne negatívne (FN):  1747
  Správne pozitívne (TP):  1033
  Návratnosť (recall):     0.3716
  Presnosť (precision):    0.3482

fault_type_8
  Správne negatívne (TN):  50633
  Falošne pozitívne (FP):  1137
  Falošne negatívne (FN):  810
  Správne pozitívne (TP):  1798
  Návratnosť (recall):     0.6894
  Presnosť (precision):    0.6126

fault_type_9
  Správne negatívne (TN):  49904
  Falošne pozitívne (FP):  1406
  Falošne negatívne (FN):  903
  Správne pozitívne (TP):  2165
  Návratnosť (recall):     0.7057
  Presnosť (precision):    0.6063

fault_type_13
  Správne negatívne (TN):  52055
  Falošne pozitívne (FP):  1372
  Falošne negatívne (FN):  951
  Správne pozitívne (TP):  0
  Návratnosť (recall):     0.0000
  Presnosť (precision):    0.0000